In [2]:
# Import Block
!pip install qiskit
!pip install qiskit_aer
!pip install qiskit.circuit.library
import random
import math
import numpy as np
import qiskit as qc
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Operator, Statevector

ERROR: Could not find a version that satisfies the requirement qiskit.circuit.library (from versions: none)
ERROR: No matching distribution found for qiskit.circuit.library


In [3]:
# Internal Function Definitions
def DimensionGiver(n):
  N = n
  Possible_Pairs = []
  for i in range(2,N):
    if (N % i == 0):
      if (i <= (N / i)):
        Possible_Pairs.append([i, int(N / i)])
      else:     #This Stuff is so that we define width as the smaller value, and height as larger value to consistantly be able to compute
        Possible_Pairs.append([int(N / i), i])
  Dimension = random.choice(Possible_Pairs)
  return Dimension


def GiveHammingWeight(x):
  BitX = bin(x)[2:]
  return BitX.count("1")

def GiveParity(x):
  return GiveHammingWeight(x) % 2

def KeyChoice(dimension):
  x, y = dimension
  target_parity = GiveParity(x) ^ GiveParity(y)

  repeat = True
  while repeat:
    key_x = random.randrange(2, x+1, 2)  #We do this now, because out lord and saviour, PDF, has told us to use even superposition :D
    key_y = random.randint(1, y)
    if (GiveParity(key_x) ^ GiveParity(key_y)) == target_parity:
      repeat = False

  return [key_x, key_y]


def GiveQubitCount(dimension):
  x, y = dimension
  QubitCount = [int(np.ceil(np.log2(x))),int(np.ceil(np.log2(y)))]
  return QubitCount

In [4]:
# Automated Space Preparation
def Initiate():
  NumberChoices = [4, 6, 9, 10, 14, 15, 21, 22, 25, 26, 33, 34, 35, 38, 39, 46, 49, 51, 55, 57, 58, 62, 65, 69, 74, 77, 82, 85, 86, 87, 91, 93, 94, 95, 106, 111, 115, 118, 119, 121, 122, 123, 129, 133, 134, 141, 142, 143, 145, 146, 155, 158, 159, 161, 166, 169, 177, 178, 183, 185, 187, 194, 201, 202, 203, 205, 206, 209, 213, 214, 215, 217, 218, 219, 221, 226, 235, 237, 247, 249, 253, 254]
  N = random.choice(NumberChoices)
  Dimension = DimensionGiver(N)
  print(f"Dimension: {Dimension}")
  print(f"Key : {KeyChoice(Dimension)}")
  
  register_A = QuantumRegister(GiveQubitCount(Dimension)[0], name='Width')
  register_B = QuantumRegister(GiveQubitCount(Dimension)[1], name='Height')
  ancilla = QuantumRegister(3, name='Ancilla')
  
  circuit = QuantumCircuit(register_A, register_B, ancilla)
  
  print(f"Given N: {N}")
  return circuit, N


#Here, build unitary matrix
def BuildUnitary(Gamma, N):
  n = int(np.ceil(np.log2(N)))
  M = 2**n

  U = np.zeros((M, M))

  for y in range(M):
    if (y < N):
      output = (Gamma * y) % N
    else:
      output = y
    U[output][y] = 1
  return Operator(U)


#The function which runs iterations of unitary until the invariant is found
def RunIter(n,U,N,Y):

  Y_int = int(Y,2)
  print(f"Y = {Y_int}")

  qc = QuantumCircuit(n)
  for i in range(len(Y)):
          if Y[i] == '1':
              qc.x(len(Y) - 1 - i)

  for i in range(N):

    qc.unitary(U, range(n))
    Y_new = Statevector.from_instruction(qc)
    if ((np.argmax(np.abs(Y_new.data)) == Y_int) & (i > 1)):
      return (i+1)


#The master Call function which, given a total dimension N, gives the dimensions of the space
def ProvideDimensions(N):
    n = int(np.ceil(np.log2(N)))

    valid_Gamma = []
    for i in range(2, N):
        if math.gcd(i, N) == 1:
            valid_Gamma.append(i)

    Gamma = random.choice(valid_Gamma)
    U = BuildUnitary(Gamma, N)

    Y_val = random.randint(1, N)
    Y = format(Y_val, 'b')

    r = RunIter(n, U, N, Y)
    if r is None:
        return None

    x = pow(int(Gamma), r // 2, int(N))
    if x == 1:
        return None

    Final1 = math.gcd(N, int(x + 1))
    Final2 = math.gcd(N, int(x - 1))
    for f in (Final1, Final2):
        if f > 1 and N % f == 0:
            other = N // f
            if other > 1:
                return (f, other)

    return None



In [5]:
Space, N = Initiate()

dims = None
while dims is None:
    dims = ProvideDimensions(N)
Dim1, Dim2 = dims

if(Dim1 <= Dim2):
  Width = Dim1
  Height = Dim2
else:
  Width = Dim2
  Height = Dim1

print(f"Width is: {Width}, and Height is: {Height}")


Dimension: [2, 31]
Key : [2, 21]
Given N: 62
Y = 26
Width is: 2, and Height is: 31


Phase 1 Complete, moving on to phase 2

In [6]:
#Defining the function which creates even superposition
from qiskit.circuit.library import MCPhaseGate as MCPGate
def CreateConstrainedEvenSuperposition(Space, N, width):
  n = int(np.ceil(np.log2(N)))
  # Exclude qubit 0 (the LSB) so it stays |0>, forcing every x in the
  # superposition to be even -- matching KeyChoice's even key_x constraint.
  # (Previously excluded qubit (width-1), the MSB, which only restricted
  # x's magnitude and had nothing to do with parity/evenness.)
  targets = [q for q in range(n) if q != 0]
  Space.h(targets)
  return Space


width_qubits, height_qubits = GiveQubitCount([Width, Height])
search_qubits_count = width_qubits + height_qubits

EvenSpace = CreateConstrainedEvenSuperposition(Space, N, width_qubits)
EvenSpace.draw('text')

#Remember to change how key is chosen to fit the HW(x) XOR HW(y) = HW(W|H)
#Remember to check only the parity
#Remember to use minimum ancilla
#Remember that the purpose of phase 2 is only too reduce states by a factor of 2, phase 3 finds keystate

def ApplyHammingWeightResonanceOracle(Space, width, n, target_parity):
    anc_px, anc_py, target = n, n + 1, n + 2
    x_qubits = list(range(0, width))
    y_qubits = list(range(width, n))

    for q in x_qubits:
        Space.cx(q, anc_px)
    for q in y_qubits:
        Space.cx(q, anc_py)

    #combine both parities into anc_px: anc_px now = HW(x) + HW(y)
    Space.cx(anc_py, anc_px)

    # if target==1, flip anc_px once so "1" always means "condition holds"
    if target_parity == 1:
        Space.x(anc_px)

    Space.cx(anc_px, target)

    if target_parity == 1:
        Space.x(anc_px)
    Space.cx(anc_py, anc_px)
    for q in reversed(y_qubits):
        Space.cx(q, anc_py)
    for q in reversed(x_qubits):
        Space.cx(q, anc_px)

    return Space
def build_pi3_diffuser(num_search_qubits):

    qc = QuantumCircuit(num_search_qubits, name='D_π/3')
    
    qc.h(range(num_search_qubits))
    qc.x(range(num_search_qubits))
    
    mcp = MCPGate(np.pi / 3, num_ctrl_qubits=num_search_qubits - 1)
    qc.append(mcp, list(range(num_search_qubits)))
    
    qc.x(range(num_search_qubits))
    qc.h(range(num_search_qubits))
    
    return qc.to_instruction()


key = KeyChoice((Width, Height))
print(f"Key: {key}")
print(GiveHammingWeight(key[0]) % 2, GiveHammingWeight(key[1]) % 2)



EvenSpace.x(search_qubits_count + 2)
EvenSpace.h(search_qubits_count + 2)



OracleSpace = ApplyHammingWeightResonanceOracle(EvenSpace, width_qubits, search_qubits_count, 1)
diffuser = build_pi3_diffuser(search_qubits_count)
OracleSpace.append(diffuser, list(range(search_qubits_count)))
import random
for _ in range(30):
  dim = (random.randint(2,20), random.randint(2,20))
  print(dim, KeyChoice(dim))

Key: [2, 13]
1 1
(17, 4) [10, 1]
(6, 12) [6, 6]
(12, 3) [4, 2]
(10, 13) [2, 3]
(3, 14) [2, 9]
(12, 15) [12, 10]
(9, 7) [2, 6]
(15, 12) [12, 9]
(14, 20) [10, 4]
(17, 12) [12, 9]
(13, 20) [10, 8]
(18, 14) [12, 11]
(7, 9) [6, 1]
(9, 2) [6, 1]
(10, 11) [8, 5]
(11, 6) [2, 5]
(13, 9) [12, 7]
(13, 12) [2, 3]
(9, 2) [6, 2]
(9, 5) [2, 4]
(18, 13) [4, 6]
(11, 15) [8, 9]
(18, 7) [6, 7]
(16, 7) [6, 6]
(10, 16) [8, 6]
(14, 4) [10, 3]
(3, 16) [2, 10]
(19, 11) [18, 6]
(3, 7) [2, 5]
(17, 17) [4, 8]


In [7]:
# Measure the search qubits and decode the result back into (x, y)
from qiskit_aer import AerSimulator
from qiskit import transpile, ClassicalRegister

# Measure only the x/y search qubits, not the ancillas
sim_circuit = OracleSpace.copy()
creg = ClassicalRegister(search_qubits_count, name='result')
sim_circuit.add_register(creg)
sim_circuit.measure(list(range(search_qubits_count)), creg)

simulator = AerSimulator()
compiled = transpile(sim_circuit, simulator)
result = simulator.run(compiled, shots=2048).result()
counts = result.get_counts()

# Qiskit reports bitstrings with qubit (n-1) leftmost and qubit 0 rightmost,
# so reverse before slicing out the x-register and y-register bits.
def decode_bitstring(bitstring, width_qubits, height_qubits):
    bits = bitstring[::-1]
    x_bits = bits[0:width_qubits]
    y_bits = bits[width_qubits:width_qubits + height_qubits]
    x_val = int(x_bits[::-1], 2)
    y_val = int(y_bits[::-1], 2)
    return x_val, y_val

decoded_counts = {}
for bitstring, count in counts.items():
    xy = decode_bitstring(bitstring, width_qubits, height_qubits)
    decoded_counts[xy] = decoded_counts.get(xy, 0) + count

sorted_results = sorted(decoded_counts.items(), key=lambda kv: -kv[1])

print(f"Target key: {key}")
print("Top measured (x, y) pairs:")
for (x_val, y_val), count in sorted_results[:10]:
    marker = " <-- TARGET" if [x_val, y_val] == key else ""
    print(f"  ({x_val}, {y_val}): {count} shots{marker}")


Target key: [2, 13]
Top measured (x, y) pairs:
  (0, 19): 81 shots
  (0, 27): 79 shots
  (0, 24): 77 shots
  (0, 22): 75 shots
  (0, 15): 71 shots
  (0, 16): 71 shots
  (0, 3): 70 shots
  (0, 14): 69 shots
  (0, 6): 69 shots
  (0, 2): 68 shots


In [ ]:

from qiskit_ibm_runtime import QiskitRuntimeService

# Uncomment and run once, then comment back out:
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token="cL3TvydWUlcyfKhqUpfuM8RP6q3A9msBA40JCzWmAt9g",
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/459831883ef8429d9a9f0d1d67638fc1:0d42dff3-618a-41f6-8844-281911b05288::",
    set_as_default=True,
    overwrite=True
)
print(QiskitRuntimeService.saved_accounts())



{'default-ibm-quantum-platform': {'channel': 'ibm_quantum_platform', 'url': 'https://cloud.ibm.com', 'token': 'cL3TvydWUlcyfKhqUpfuM8RP6q3A9msBA40JCzWmAt9g', 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/459831883ef8429d9a9f0d1d67638fc1:0d42dff3-618a-41f6-8844-281911b05288::', 'verify': True, 'private_endpoint': False}}


In [ ]:
# Run OracleSpace on real IBM Quantum hardware
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit import ClassicalRegister

service = QiskitRuntimeService(channel="ibm_quantum_platform")

backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=search_qubits_count + 3
)
print(f"Running on: {backend.name}")

hw_circuit = OracleSpace.copy()
creg = ClassicalRegister(search_qubits_count, name='result')
hw_circuit.add_register(creg)
hw_circuit.measure(list(range(search_qubits_count)), creg)

pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuit = pm.run(hw_circuit)
print(f"Transpiled depth: {isa_circuit.depth()}, gate count: {isa_circuit.size()}")

# Submit the job
sampler = SamplerV2(mode=backend)
job = sampler.run([isa_circuit], shots=2048)
print(f"Job ID: {job.job_id()}")
print(f"Job status: {job.status()}")


Running on: ibm_fez
Transpiled depth: 574, gate count: 909
Job ID: d94ho55gc6cc73ff4ib0
Job status: QUEUED


In [16]:

hw_result = job.result()
hw_counts = hw_result[0].data.result.get_counts()

decoded_hw_counts = {}
for bitstring, count in hw_counts.items():
    xy = decode_bitstring(bitstring, width_qubits, height_qubits)
    decoded_hw_counts[xy] = decoded_hw_counts.get(xy, 0) + count

sorted_hw_results = sorted(decoded_hw_counts.items(), key=lambda kv: -kv[1])

print(f"Target key: {key}")
print("Top measured (x, y) pairs on real hardware:")
for (x_val, y_val), count in sorted_hw_results[:10]:
    marker = " <-- TARGET" if [x_val, y_val] == key else ""
    print(f"  ({x_val}, {y_val}): {count} shots{marker}")


Target key: [2, 13]
Top measured (x, y) pairs on real hardware:
  (0, 0): 84 shots
  (0, 4): 82 shots
  (0, 5): 75 shots
  (0, 20): 70 shots
  (0, 9): 65 shots
  (0, 12): 64 shots
  (0, 1): 62 shots
  (0, 16): 60 shots
  (0, 24): 59 shots
  (0, 17): 53 shots
